# Demo: LLM and token basics

```yaml
title: "Demo: LLM and token basics"
keywords: tokens, tokenizer, context window, temperature, sampling, cost, claude sonnet, teacher demo
estimated duration: 60
```

> **Lesson:** L01. Teacher-led demo notebook — projected during class, no student participation.
> Maps one-to-one to the four demos in the roadmap's
> [demos_or_activities.md](../../../../docs/origin/lesson_roadmaps/L01/demos_or_activities.md).
> **Anchor model: Claude Sonnet 4.6 (200k-token window).**
>
> Cells that call the API are guarded so the notebook runs end-to-end without a key; set
> `ANTHROPIC_API_KEY` to see them live.

## Setup

Imports + small inline helpers. Runs offline, no key needed.

In [ ]:
import tiktoken

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message

# tiktoken is OpenAI's tokenizer. We use it to SEE token boundaries, because
# Claude's API returns a token *count* but not the individual pieces. The wider
# point of Demo 1 is exactly this: tokenizers are a *family* of choices.
ENGLISH = "The quick brown fox jumps over the lazy dog."
CODE = "def f(x): return x ** 2"
JSON = '{"user_id": 12345, "events": ["click", "scroll"]}'
NON_LATIN = "こんにちは、世界"  # "hello, world" in Japanese

STRINGS = {"english": ENGLISH, "code": CODE, "json": JSON, "non_latin": NON_LATIN}


def token_pieces(text: str, encoding_name: str = "cl100k_base") -> list[str]:
    """Decode each token of `text` on its own, so we can see the boundaries.

    Example: token_pieces("def f(x)") -> ['def', ' f', '(x', ')']
    """
    enc = tiktoken.get_encoding(encoding_name)
    return [enc.decode([token_id]) for token_id in enc.encode(text)]


def show_boundaries(text: str, encoding_name: str = "cl100k_base") -> str:
    """Render token boundaries joined by '|' for projection."""
    return "|".join(token_pieces(text, encoding_name))


def tiktoken_count(text: str, encoding_name: str = "cl100k_base") -> int:
    return len(tiktoken.get_encoding(encoding_name).encode(text))


print("setup ready")

In [ ]:
# Live cells below talk to the real API. They are guarded by HAS_KEY so this
# notebook still runs top-to-bottom without a key (live cells print a skip note).
import os

HAS_KEY: bool = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("ANTHROPIC_API_KEY set:", HAS_KEY)

## Demo 1 — Tokens are not characters, words, or syllables

**Goal:** a token is whatever the tokenizer says it is. Ask the room (rhetorically) which string is
longest in tokens *before* revealing the counts.

In [ ]:
# Reveal the boundaries the model sees (visualized via tiktoken).
for name, text in STRINGS.items():
    print(f"[{name}] {len(text)} chars -> {tiktoken_count(text)} tokens")
    print("   ", show_boundaries(text))
    print()

In [ ]:
# Tokenizers are a FAMILY: the same string counts differently across vendors.
# cl100k_base (GPT-3.5/4) vs o200k_base (GPT-4o). Claude has its own tokenizer too,
# whose real count shows up in `usage` on any live call (see Demo 4).
for name, text in STRINGS.items():
    cl = tiktoken_count(text, "cl100k_base")
    try:
        o2 = tiktoken_count(text, "o200k_base")
    except Exception:
        # An older tiktoken may not ship the o200k_base vocab; -1 marks "unavailable".
        o2 = -1
    print(f"[{name:9}] cl100k={cl:3}  o200k={o2:3}")

In [ ]:
# The rule of thumb (~4 chars/token) holds for English and breaks on JSON.
for name in ("english", "json"):
    text = STRINGS[name]
    ratio = len(text) / tiktoken_count(text)
    print(f"[{name}] {ratio:.1f} chars/token  (rule of thumb says ~4.0)")

## Demo 2 — The context window is one shared budget

**Goal:** input + system + history + output all draw from the *same* 200k-token budget, and there
are three failure modes when you run out. We use numbers (no live call) so the ceiling is explicit.

In [ ]:
WINDOW = 200_000  # Claude Sonnet 4.6 standard window, in tokens


def window_meter(
    system: int, history: int, current: int, reserved_output: int, width: int = 40
) -> str:
    """ASCII meter of how a 200k window is divided. Each block ~= WINDOW/width tokens."""
    used = system + history + current + reserved_output
    per_block = WINDOW / width
    filled = min(width, round(used / per_block))
    bar = "█" * filled + "░" * (width - filled)
    return f"[{bar}] {used:,}/{WINDOW:,} tokens used"


# Case 1: tiny call — barely moves the meter.
print("tiny call    ", window_meter(system=20, history=0, current=15, reserved_output=200))
# Case 2: long history — the same prompt, 20 prior turns of ~500 tokens. Fits, but fills.
print("with history ", window_meter(system=20, history=20 * 500, current=15, reserved_output=200))
# Case 3: overflow — a 250k-token blob prepended. Exceeds the window.
overflow = 20 + 250_000 + 15 + 200
print("overflow     ", window_meter(system=20, history=250_000, current=15, reserved_output=200))
print(
    f"overflow total = {overflow:,} tokens > {WINDOW:,} window  -> the API would HARD-REJECT this"
)

**Three failure modes** (say these out loud while the overflow number is on screen):

1. **Hard rejection** — what the number above shows: the API refuses before running.
2. **Silent truncation** — some clients/frameworks quietly drop the oldest turns. The call
   succeeds but the model "forgot." *This is the dangerous one.*
3. **Quality degradation** — even when it fits, models can lose track of material buried
   mid-context. Foreshadows L15 (context management) and L17 (RAG).

## Demo 3 — Temperature is a knob, not a switch

**Goal:** same prompt, different temperatures, different spread — and temperature 0 is *low*
variance, not *zero*. Show the distribution diagram (slide) before running anything.

In [ ]:
COFFEE_PROMPT = "Suggest a name for a coffee shop on a rainy street in Seattle. Just the name, one or two words."


def sample_n(temperature: float, n: int) -> list[str]:
    """Run the same prompt n times at a given temperature; return the replies."""
    if not HAS_KEY:
        return [f"<needs ANTHROPIC_API_KEY> temp={temperature}"] * n
    client = AnthropicClient()
    out: list[str] = []
    for _ in range(n):
        reply = client.chat([Message.user(COFFEE_PROMPT)], max_tokens=16, temperature=temperature)
        out.append(reply.text.strip())
    return out


print("temperature=0 (expect near-identical):")
for line in sample_n(0.0, 5):
    print("  ", line)

In [ ]:
print("temperature=1 (expect variety):")
for line in sample_n(1.0, 5):
    print("  ", line)

In [ ]:
# Punchline: run temp=0 MORE times and watch for the rare disagreement.
print("temperature=0, 10 more runs (look for any that differ):")
for line in sample_n(0.0, 10):
    print("  ", line)

- Temperature reshapes the distribution *before* sampling. Low = the top token dominates.
- `top_p` (nucleus) and `top_k` restrict the *candidate set* instead — different lever. We name
  them but don't tune them in this course.
- Temperature 0 ⇒ low variance, not zero: floating-point math, batching, and tie-breaks leak through.

## Demo 4 — Cost is per token, both directions, every call

**Goal:** real dollar amounts, the input/output asymmetry, and the conversation-history staircase.

In [ ]:
# Illustrative rates only. ALWAYS confirm current Claude Sonnet pricing on Anthropic's
# pricing page on the day you teach — a stale slide undermines every cost claim that follows.
INPUT_USD_PER_MTOK = 3.00
OUTPUT_USD_PER_MTOK = 15.00  # note: output is the EXPENSIVE direction (~3-5x input)


def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    """Cost of one call in USD, given token counts and per-MILLION-token rates."""
    return (
        input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
        + output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    )


# Asymmetry: 2k-in/50-out vs 50-in/2k-out. The second costs more.
print("long input,  short answer:", f"${call_cost_usd(2000, 50):.5f}")
print("short input, long answer: ", f"${call_cost_usd(50, 2000):.5f}  <- output is pricier")

In [ ]:
# The conversation-history staircase: each turn re-sends everything before it.
PER_TURN = 200  # new tokens added per turn
turns = 5
cumulative = 0
total_cost = 0.0
for t in range(1, turns + 1):
    cumulative += PER_TURN  # this turn's new tokens are now part of history
    out_tokens = 60  # a short reply each turn
    cost = call_cost_usd(cumulative, out_tokens)
    total_cost += cost
    print(f"turn {t}: input re-sent ~{cumulative:4} tokens  -> ${cost:.5f}")
print(f"session total: ${total_cost:.5f}  (history is re-billed every turn)")

In [ ]:
# Order-of-magnitude ladder: one call -> agent run -> dev iteration -> small deployment.
one_call = call_cost_usd(2000, 300)
print(f"1 call                        ~ ${one_call:.5f}")
print(f"x10  (a 10-step agent run)    ~ ${one_call * 10:.4f}")
print(f"x100 (dev iteration)          ~ ${one_call * 10 * 100:.2f}")
print(f"x1000 (small deployment)      ~ ${one_call * 10 * 100 * 1000:,.2f}")
print("\nThis jump from 'free' to 'a real budget' is when cost becomes a design concern.")

- Output tokens cost more than input — long prompt + short answer is often cheaper than the reverse.
- The history staircase is the #1 self-inflicted cost surprise. Prompt caching (L15) and retrieval
  (L17) exist to fight it.
- **Bridge to L02:** every prompt is a budget decision — tokens, dollars, and window, all at once.